<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Ejercicios/08_Taller_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Taller práctico: atención, Transformers y modelos preentrenados</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Instrucciones</h2>

Este taller practica los temas de `16_Atencion_Transformers.ipynb`, `17_BERT_GPT_Fine_Tuning.ipynb` y `18_Aplicaciones_Preentrenados.ipynb` con datos y tareas distintos a los de las notas de clase:

1. Implementarás la multi-head attention desde cero y la compararás con la de **PyTorch**, cuyas convenciones de máscaras son distintas a las de Keras, y medirás su costo cuadrático.
2. Construirás un bloque Transformer en PyTorch y comprobarás, con una tarea sintética diseñada para ello, **por qué** hace falta la codificación posicional.
3. Harás fine-tuning de dos modelos preentrenados en español para clasificar el **sentimiento de tuits** (positivo, neutro, negativo) y medirás el efecto del **dominio de preentrenamiento** y de la **semilla aleatoria**.
4. Evaluarás un modelo de resumen en un **dominio distinto** al de su entrenamiento y calibrarás un modelo de respuesta a preguntas para que **se abstenga** cuando la respuesta no está en el texto.

> **GPU recomendada.** En Google Colab activa una GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*. Con GPU, todo el taller se ejecuta en unos 10 a 15 minutos.

Resuelve cada ejercicio en la celda indicada y responde las preguntas en celdas markdown.

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Ejercicio 1. Self-attention desde cero</span>](#ejercicio-1)
- [<span style="color: #005C99;">Ejercicio 2. Bloque Transformer en PyTorch</span>](#ejercicio-2)
- [<span style="color: #005C99;">Ejercicio 3. Fine-tuning de BERT</span>](#ejercicio-3)
- [<span style="color: #005C99;">Ejercicio 4. Sumarización y QA</span>](#ejercicio-4)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib seaborn scikit-learn datasets "transformers>=5" accelerate torch sentencepiece rouge_score

In [ ]:
import os
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import transformers
from transformers import AutoTokenizer, pipeline, set_seed
import re
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
set_seed(SEED)  # fija las semillas de Python, NumPy y PyTorch
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers {transformers.__version__} | PyTorch {torch.__version__} | dispositivo: {dispositivo}")

---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Self-attention desde cero</h2>

**Objetivo:** implementar la multi-head self-attention con máscaras, verificarla contra `torch.nn.MultiheadAttention` y medir su costo en función de la longitud.

1. Implementa en NumPy `multi_head(X, W_in, b_in, W_out, b_out, h, mascara_causal=False, relleno=None)` que replique a `torch.nn.MultiheadAttention` (con `batch_first=True`) para **una** secuencia `X` de forma `(T, d)`:
   - `W_in` tiene forma `(3d, d)` y contiene, apiladas, las matrices de consultas, claves y valores; cada una se aplica como `X @ W.T + b`.
   - Cada cabeza usa un bloque consecutivo de $d/h$ columnas de $Q$, $K$ y $V$.
   - `relleno` es un vector booleano de longitud $T$ con `True` en las posiciones de **relleno**, a las que nadie debe atender.
   - La función debe devolver la salida `(T, d)` y los pesos de atención `(h, T, T)`.
2. Verifica tu implementación contra `nn.MultiheadAttention(d, h, batch_first=True)` en tres casos: sin máscara, con máscara causal y con relleno en las 2 últimas posiciones. Atención: en PyTorch, `attn_mask` y `key_padding_mask` marcan con `True` las posiciones **prohibidas**, al revés que la máscara de Keras del cuaderno 16. Compara también los pesos (usa `need_weights=True, average_attn_weights=False`).
3. Mide el tiempo de tu función con una sola cabeza, `d = 64`, para $T \in \{256, 512, 1024, 2048, 4096\}$, tomando el mínimo de 3 repeticiones. Para que la medición sea estable, limita NumPy a un solo hilo con `threadpoolctl.threadpool_limits(1)`. Grafica en escala log-log y estima la pendiente con los tres valores más grandes. ¿Es cuadrática? ¿Cuánta memoria ocupa la matriz de pesos con $T = 100\,000$ (en float32)?

In [ ]:
# Escribe tu código aquí
def multi_head(X, W_in, b_in, W_out, b_out, h, mascara_causal=False, relleno=None):
    pass

---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. Bloque Transformer en PyTorch</h2>

**Objetivo:** implementar un bloque codificador del Transformer en PyTorch y demostrar experimentalmente que, sin codificación posicional, no puede resolver una tarea que depende del orden.

Usaremos una tarea sintética diseñada para que **solo el orden** importe. Cada secuencia tiene 24 tokens: un token `A`, un token `B` y 22 tokens de relleno al azar. La etiqueta es 1 si `A` aparece **antes** que `B`. La celda de código incluye la función que genera los datos (20 000 secuencias de entrenamiento y 4 000 de validación).

1. Implementa `BloqueCodificador(nn.Module)` con `nn.MultiheadAttention` (con `batch_first=True`), conexiones residuales, `LayerNorm` y una red *feed-forward* de dos capas, como en el diagrama del cuaderno 16.
2. Implementa un clasificador `Clasificador(nn.Module)` con: embedding de tokens (dimensión 64), **opcionalmente** un embedding de posiciones aprendido, 0 o más bloques codificadores, promedio sobre la secuencia y una capa lineal de salida a 2 clases.
3. Entrena tres modelos con Adam (tasa $10^{-3}$, lotes de 128, 10 épocas) y reporta la exactitud en validación:
   - (a) **Promedio de embeddings** (sin bloques ni posiciones),
   - (b) **Transformer sin posiciones** (2 bloques, 4 cabezas),
   - (c) **Transformer con posiciones** (2 bloques, 4 cabezas).
4. Explica los resultados a partir de la propiedad de **equivarianza a permutaciones** del cuaderno 16. Para el modelo (c), muestra el mapa de atención del último bloque en un ejemplo: ¿a qué tokens atiende la mayoría de las posiciones?

In [ ]:
# Tarea sintética: secuencias de 24 tokens con un "A" (id 0) y un "B" (id 1) en posiciones al azar;
# el resto son tokens de relleno al azar (ids 2 a 21). Etiqueta: 1 si A aparece antes que B.
def generar(n, T=24, V=22, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.integers(2, V, size=(n, T))
    y = np.zeros(n, dtype=np.int64)
    for i in range(n):
        pa, pb = rng.choice(T, size=2, replace=False)
        X[i, pa], X[i, pb] = 0, 1
        y[i] = int(pa < pb)
    return torch.tensor(X), torch.tensor(y)


X_sint_train, y_sint_train = generar(20_000, seed=1)
X_sint_val, y_sint_val = generar(4_000, seed=2)
print(X_sint_train[:2], y_sint_train[:2])


# Escribe tu código aquí
class BloqueCodificador(nn.Module):
    pass

---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. Fine-tuning de BERT</h2>

**Objetivo:** ajustar modelos preentrenados en español para clasificar el sentimiento de tuits, comparando un modelo de dominio general con uno preentrenado en tuits y midiendo la variabilidad entre semillas.

Usaremos la parte en español del corpus [TweetSentiment Multilingual](https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual) (Barbieri et al., 2022): 1 839 tuits de entrenamiento, 324 de validación y 870 de prueba, etiquetados como negativo (0), neutro (1) o positivo (2).

```python
base = "hf://datasets/cardiffnlp/tweet_sentiment_multilingual/data/spanish/"
tuits = load_dataset("json", data_files={s: base + f"{s}.jsonl" for s in ["train", "validation", "test"]})
```

1. Explora los datos (distribución de clases, longitud, ejemplos) y entrena una línea base TF-IDF + regresión logística. Usa el **F1 macro** como métrica principal.
2. Haz fine-tuning de dos modelos *uncased* (pasa los textos a minúsculas):
   - `dccuchile/bert-base-spanish-wwm-uncased` (BETO, preentrenado con Wikipedia, noticias, libros...),
   - `pysentimiento/robertuito-base-uncased` (RoBERTuito, preentrenado con 500 millones de tuits en español).

   Usa `Trainer` con 4 épocas, tasa de aprendizaje $5 \times 10^{-5}$, lotes de 16 y `max_length=128`. Repite cada fine-tuning con **3 semillas** distintas y reporta la media y la desviación estándar del F1 macro en **validación**.
3. ¿Cuánto varía el resultado entre semillas? ¿Es mayor o menor que la diferencia entre los dos modelos? ¿Qué implica para comparar modelos con un solo entrenamiento?
4. Elige el mejor modelo según validación y evalúa sus 3 semillas en **prueba**. Muestra la matriz de confusión de una de ellas: ¿qué clase es la más difícil?

In [ ]:
# Escribe tu código aquí

---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. Sumarización y QA</h2>

**Objetivo:** evaluar un modelo de resumen fuera de su dominio de entrenamiento y calibrar un modelo de respuesta a preguntas para que se abstenga cuando no hay respuesta.

**Parte A: resumen fuera de dominio.** El modelo `csebuetnlp/mT5_multilingual_XLSum` del cuaderno 18 se ajustó con noticias de la BBC. Evalúalo con 50 noticias de **El País** del conjunto de prueba de [MLSUM](https://huggingface.co/datasets/reciTAL/mlsum) (Scialom et al., 2020):

```python
mlsum = load_dataset("parquet", data_files={"test": "hf://datasets/reciTAL/mlsum@refs/convert/parquet/es/test/0000.parquet"})["test"]
```

1. Calcula ROUGE-1, ROUGE-2 y ROUGE-L (con un tokenizador que respete las tildes, como en el cuaderno 18) del modelo y de las líneas base *lead-1* y *lead-3* (primera oración y primeras tres oraciones).
2. Compara con los resultados del cuaderno 18 en XL-Sum. ¿El modelo sigue superando a las líneas base? Compara la longitud media de los resúmenes generados y de referencia, y lee tres ejemplos: ¿en qué se diferencian los resúmenes de referencia de MLSUM y de XL-Sum?

**Parte B: saber cuándo no responder.** Usa el modelo de QA del cuaderno 18 (`mrm8488/distill-bert-base-spanish-wwm-cased-finetuned-spa-squad2-es`) y su función `responder`, que devuelve la respuesta, el puntaje del mejor fragmento y el puntaje "sin respuesta".

3. Toma 400 preguntas de prueba de SQAC. Para la mitad, usa su contexto original (**con respuesta**); para la otra mitad, empareja cada pregunta con el contexto de **otra** pregunta elegida al azar (**sin respuesta**, en la práctica).
4. Define la diferencia $\Delta = $ puntaje del fragmento $-$ puntaje sin respuesta. Grafica la distribución de $\Delta$ en los dos grupos.
5. Con la mitad de los ejemplos (calibración), elige el umbral de $\Delta$ que maximiza la exactitud de la decisión "responder / abstenerse". Con la otra mitad, reporta esa exactitud y el F1 de las respuestas, contando como correcta una abstención en una pregunta sin respuesta y como incorrecta una respuesta en una pregunta sin respuesta. Compara con no abstenerse nunca.

In [ ]:
# Escribe tu código aquí

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Tunstall, L., von Werra, L. & Wolf, T. (2022). *Natural Language Processing with Transformers*. O'Reilly Media.
- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Vaswani, A. et al. (2017). Attention Is All You Need. *NeurIPS*.
- Dao, T. et al. (2022). FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness. *NeurIPS*.
- Barbieri, F., Espinosa Anke, L. & Camacho-Collados, J. (2022). XLM-T: Multilingual Language Models in Twitter for Sentiment Analysis and Beyond. *LREC*.
- Pérez, J. M. et al. (2022). RoBERTuito: a pre-trained language model for social media text in Spanish. *LREC*.
- Gururangan, S. et al. (2020). Don't Stop Pretraining: Adapt Language Models to Domains and Tasks. *ACL*.
- Dodge, J. et al. (2020). Fine-Tuning Pretrained Language Models: Weight Initializations, Data Orders, and Early Stopping. *arXiv:2002.06305*.
- Scialom, T. et al. (2020). MLSUM: The Multilingual Summarization Corpus. *EMNLP*.
- Rajpurkar, P., Jia, R. & Liang, P. (2018). Know What You Don't Know: Unanswerable Questions for SQuAD. *ACL*.